# Exercise 01 — Colab, and the four things every lab does

MSc Finance · Investments · FHNW · Autumn 2026

Three asset classes, a century of annual returns, and the four operations that come back
every week: **describe, regress, plot, export**.

**How the afternoon is organised.** The lab has two blocks.

*Block A — tasks 1 to 3.* You write these yourself, with the two-page cheat sheet on
your desk and nothing else. No Claude, no ChatGPT, no Copilot. Everything Block A needs is
on those two pages, and its sections follow the warm-up one for one, so anything you typed
this morning is on it. The reason is not suspicion. From Exercise 02 onwards you
will spend a good part of every lab reading code an AI wrote for you, and you can only
check code you could have written yourself.

*Block B — tasks 4 and 5.* The other way round: use whatever helps.

**How to work in the notebook.** Each code cell is a stub. The `# TODO` lines are the
steps, in order; you replace each one with the code that does it. The setup cell below is
complete — run it first and leave it alone. The names it defines (`DATA_URL`, `SHEET`,
`EQUITY`, `SMALL`, `BOND`, `BILL`, `ASSETS`, `idx`, `rets`) are the ones the tasks refer to.

Run **Runtime → Restart and run all** before you trust any number in here.

*Data: annual total-return indices for U.S. asset classes, 1927–2025, from Aswath
Damodaran's data page,
[pages.stern.nyu.edu/~adamodar](https://pages.stern.nyu.edu/~adamodar/), snapshotted in
the course repository.*

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.ticker import PercentFormatter

from exercise_utils import FHNW, setup_style, load_returns, save_results
setup_style()

BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_01/data/histretSP_investments.xlsx"
SHEET = "Total Return Index_Damadoran"

EQUITY = "S&P 500 (includes dividends)"
SMALL = "US Small cap (bottom decile)"
BOND = "US T. Bond"
BILL = "3-month T.Bill"
ASSETS = [EQUITY, BOND, BILL]

idx = load_returns(DATA_URL, sheet=SHEET)
rets = load_returns(DATA_URL, sheet=SHEET, to_returns=True)

print("Setup complete.", rets.shape[0], "annual returns,",
      rets.index.min(), "to", rets.index.max())

---
# Block A — you write this

Cheat sheet only. Three tasks, roughly seventy minutes. Every number in Block A can be
checked by hand or against a number you already have, so you never need an authority to
tell you whether it is right.

## Task 1 — Five returns, on paper and in code

An asset earns +25 %, −20 %, +15 %, −5 % and +10 % over five years. Compute the arithmetic
mean, the geometric mean and the sample standard deviation on paper first, dividing by
n − 1. Then reproduce all three here and check them against your hand calculation.

*Deliverable: three printed percentages that agree with your paper.*

In [ ]:
# SOLUTION
# STUB: put the five returns into a pandas Series called `a`
a = pd.Series([0.25, -0.20, 0.15, -0.05, 0.10])

# STUB: arithmetic mean, geometric mean and sample standard deviation of `a`
arith = a.mean()
geo = (1 + a).prod() ** (1 / len(a)) - 1
sd = a.std()                      # pandas divides by n-1 by default

# STUB: print all three as percentages with two decimals
print(f"arithmetic mean {arith:7.2%}")
print(f"geometric mean  {geo:7.2%}")
print(f"std. deviation  {sd:7.2%}")

<!-- solution -->
Hand values: **5.00 %, 3.74 %, 17.68 %**. The geometric mean is the lower of the two
because a loss of 20 % takes more than a 20 % gain to undo.

Anyone who divided by n instead of n − 1 gets 15.81 % and will disagree with `pandas`,
which is the useful moment: the sample standard deviation corrects for the fact that the
mean was itself estimated from the same five numbers. `a.std()` is n − 1;
`a.std(ddof=0)` is n.

## Task 2 — Select, then describe

Keep the S&P 500, US T. Bonds and 3-month T-Bills from the Damodaran total-return indices,
and report the arithmetic mean, the geometric mean and the standard deviation for two
samples: the full history, and 1990 onwards. Write the three statistics once, as a
function, and use it twice — the signature is given.

*Deliverable: two tables, three rows and three columns each.*

In [ ]:
# SOLUTION
# STUB: keep only the three ASSETS columns of `rets`
sub = rets[ASSETS]

# STUB: fill in the three statistics; the signature is given
def stats(r):                                                        # KEEP
    return pd.DataFrame({
        "Arith. mean": r.mean(),
        "Geo. mean": (1 + r).prod() ** (1 / len(r)) - 1,
        "Std. dev.": r.std(),
    })

# STUB: apply stats() to the full sample and to 1990 onwards
full = stats(sub)
recent = stats(sub.loc[1990:])

# STUB: print both tables as percentages, each under a heading
print("1928-2025")
print(full.to_string(float_format=lambda v: f"{v:7.2%}"))
print("\n1990-2025")
print(recent.to_string(float_format=lambda v: f"{v:7.2%}"))

<!-- solution -->
Full sample: the S&P 500 returns **11.85 %** arithmetic, **10.02 %** geometric, with a
standard deviation of **19.40 %**. Bonds 4.82 / 4.53 / 7.90 %, bills 3.41 / 3.37 / 3.04 %.
The *ordering* of both mean and volatility is the same in the two samples; the *levels* are
not. Bills earn less after 1990 (2.79 %) and bonds are more volatile (9.22 %).

Note the gap between the two means: 1.84 points for the S&P 500, 0.29 for bonds, 0.04 for
bills. Volatility is what orders them, and lecture 02 gives that gap its name.

Two things go wrong here. `rets[ASSETS]` with one pair of brackets and a list gives a
DataFrame; `rets[EQUITY]` gives a Series, and `stats()` then returns nonsense rather than
an error. And `.loc[1990:]` includes 1990 — the index is the year, so this is label
slicing, where the endpoint is inside.

## Task 3 — A century, one decade at a time

Split the S&P 500 annual returns into decades, 1930–1939 through 2020–2025, and report the
arithmetic mean of each. Print the ten numbers with the full-sample mean from task 2
underneath. The series begins in 1928, so the first two years sit outside the ten decades.

*Deliverable: a table of ten decade means, and the full-sample mean below it.*

In [ ]:
# SOLUTION
# STUB: build a list called `rows`, looping over the decade start years and collecting the mean of each decade
rows = []
for y in range(1930, 2030, 10):
    window = rets[EQUITY].loc[y:y + 9]
    rows.append({"Decade": f"{y}s", "Years": len(window), "Mean": window.mean()})

# STUB: turn `rows` into a DataFrame indexed by decade
decades = pd.DataFrame(rows).set_index("Decade")

# STUB: print the decade means, then the full-sample mean underneath
print(decades.to_string(formatters={"Mean": lambda v: f"{v:7.2%}"}))
print(f"\nfull sample 1928-2025   {rets[EQUITY].mean():7.2%}")

<!-- solution -->
The ten decade means run from **1.16 %** (2000s) to **20.93 %** (1950s), a spread of almost
twenty percentage points around a full-sample mean of 11.85 %. Nothing about the underlying
series changes between decades. Only the ten years that happen to fall inside the window do.

The arithmetic behind it: with an annual standard deviation of 19.4 %, a mean computed from
ten observations carries a standard error of 19.4 / √10 ≈ **6.1** percentage points. Two
standard errors either side of the truth is a range twenty-four points wide, which is
roughly the spread the table shows. A decade is not a long sample.

This is the experiment lecture 02 runs properly, with rolling windows instead of calendar
decades. Students who take one thing from this exercise should take this.

The 2020s row has six years in it, not ten, which is why the `Years` column is in the
table at all: a mean over six observations is noisier still, and a table that hides its
sample sizes invites the reader to forget that.

---
# Block B — AI allowed

Two tasks. Task 4 is a regression you do not have to write, task 5 is the figure and the
export. Use whatever help you like from here on.

## Task 4 — Two regressions, read rather than written

The cell below regresses the excess return of US small-cap stocks on the excess return of
the S&P 500, and then does the same for US T. Bonds. Excess means in excess of the
3-month T-Bill, on both sides. The code is given: run it, then answer three questions in
the text cell underneath.

1. Small caps have a slope of 1.44. In one sentence, what does that say about how a
   small-cap portfolio behaves relative to the market?
2. Bonds have an R² of 0.00. What does that make bonds useful for?
3. The bond regression has an R² of 0.00 and yet an intercept of 1.26 % a year. If the
   market explains none of it, what is that intercept measuring?

*Deliverable: three short answers, written in the notebook.*

In [ ]:
# Given — read it, run it, interpret the output. You do not have to write this.
excess = rets[[SMALL, EQUITY, BOND]].sub(rets[BILL], axis=0)

X = sm.add_constant(excess[EQUITY])
for name in [SMALL, BOND]:
    res = sm.OLS(excess[name], X).fit()
    print(f"{name:32s} intercept {res.params.iloc[0]:6.2%}   "
          f"slope {res.params.iloc[1]:5.2f}   R2 {res.rsquared:4.2f}")

Your answers:

1.
2.
3.

<!-- solution -->
Small caps: intercept **2.21 %**, slope **1.44**, R² **0.54**. Bonds: intercept
**1.26 %**, slope **0.02**, R² **0.00**.

1. Small caps are the same bet as the market, taken 44 % harder. A market year of +10 %
   maps into roughly +16 % (= +2.21 % + 1.44 * 10.0 %) for small caps, and a market year of −10 % into roughly −12 %
   once the intercept is carried along.
2. Bonds are a *different* bet. Nothing of what happens to the S&P 500 shows up in them,
   which is exactly what a diversifier is. The correlation of the two excess returns over
   the century is 0.04.
3. The average excess return of bonds over bills, and nothing else. When the slope is
   essentially zero, the fitted line is flat and the intercept has to sit at the mean of
   the left-hand side: bonds averaged 1.41 % a year over T-Bills, and
   1.26 + 0.02 × 8.44 = 1.43 recovers it. That is the term premium, arriving as an
   intercept because the market explains none of it. Whether 1.26 % is distinguishable
   from zero is a different question, and we do not yet have the tool to ask it.

Both slopes are betas and both intercepts are alphas. We will use those names, and the
standard errors that belong with them, from lecture 06 onwards. Note what the given code
deliberately does *not* print: `res.bse` and `res.tvalues` sit on that fitted object all
along. The reason we ignore them for five more weeks is that a t-statistic without the
sampling distribution behind it is a number to recite rather than a tool — which is also
why nobody can yet say whether the 2.21 % small-cap intercept is real.

## Task 5 — One figure, and take it with you

Draw the two regressions from task 4 as a two-panel figure, then export it together with
your tables from tasks 2 and 3. You choose the axes; the aim is a figure that would stand on its own
in an exam answer.

If you finish early, add the extra figure at the end: the ten decade means as a bar chart
with the full-sample mean drawn across it. That is the picture we look at together at the
end of the session.

*Deliverable: a downloaded ZIP with one figure and three tables in it.*

In [ ]:
# SOLUTION
# STUB: one figure, two panels, sharing both axes
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharex=True, sharey=True)   # KEEP
grid = np.linspace(excess[EQUITY].min(), excess[EQUITY].max(), 50)            # KEEP

# STUB: in each panel, scatter the asset against the market and draw the fitted line
for ax, name in zip(axes, [SMALL, BOND]):
    res = sm.OLS(excess[name], sm.add_constant(excess[EQUITY])).fit()
    ax.scatter(excess[EQUITY], excess[name], s=18, alpha=0.7, color=FHNW["blue"])
    ax.plot(grid, res.params.iloc[0] + res.params.iloc[1] * grid, color="black", lw=1.4)
    ax.axhline(0, lw=0.6, color="grey")
    ax.axvline(0, lw=0.6, color="grey")
    ax.set_xlabel("S&P 500 excess return")
    ax.set_title(f"{name}  (slope {res.params.iloc[1]:.2f})", fontsize=10)

# STUB: label the shared y axis, format both axes as percentages, tidy the layout
axes[0].set_ylabel("Asset excess return")
axes[0].xaxis.set_major_formatter(PercentFormatter(xmax=1))
axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=1))
fig.tight_layout()
plt.show()

In [ ]:
# SOLUTION
# STUB: export the figure and your three tables as one ZIP
save_results(figures={"regressions": fig},
             tables={"descriptives_full": full,
                     "descriptives_1990": recent,
                     "decades": decades},
             name="ex01")

### Optional — the decade figure

Only if you have time. It downloads a second ZIP of its own, so it does not matter whether
you get this far.

In [ ]:
# SOLUTION
# STUB: bar chart of the decade means, with the full-sample mean drawn across it
fig_dec, ax = plt.subplots(figsize=(8, 4.2))
ax.bar(decades.index, decades["Mean"], color=FHNW["blue"])
ax.axhline(rets[EQUITY].mean(), color=FHNW["red"], lw=1.4)
ax.set_xlabel("Decade")
ax.set_ylabel("Mean annual return, S&P 500")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=1))
plt.show()

# STUB: export this figure too
save_results(figures={"decade_means": fig_dec}, name="ex01_decades")

<!-- solution -->
The two panels make the regression output visible: the small-cap cloud is steep and wide,
the bond cloud is a formless blob with no tilt. R² is the tightness of the cloud around the
line, and the reason a single number is worth plotting before it is trusted.

The bar chart is the figure to keep. Ten bars, one horizontal line, and the whole of
lecture 02's estimation-risk argument sitting in the distance between them.

Two export traps. `save_results` wants the **figure** object, not the axes — pass `fig`,
not `ax`. And if nothing downloads, pop-ups are blocked for the Colab domain: allow them
and run the cell again.

---
## Where this goes next

This notebook, with the solutions filled in, appears in Moodle at the end of the session.
Keep the cheat sheet: it is the reference for the whole semester, and every lab from here
on assumes the twelve or so commands on it.

Two quizzes are open in Moodle until Sunday: six drill items and one open question. Both
are ungraded; both are exactly the format the exams use.

Lecture 02 picks up tasks 2 and 3 directly. The gap between the two means becomes variance
drag, and the spread of the decade means becomes estimation risk.